# Jeonse-Only Prediction: No Monthly-Rent Conversion, No Scaler

This Colab notebook runs the Jeonse prediction experiment using **Jeonse rows only**. Monthly-rent rows are removed, so no monthly-rent-to-Jeonse conversion is applied. Numerical features are passed through without `StandardScaler` or `RobustScaler`, following the requested no-scaler setup.

Target variable: `보증금(만원)` from Jeonse transactions only.

## 1. Colab Setup

The notebook clones the same review branch so every teammate runs the same code and data version.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import importlib.util

REPO_URL = "https://github.com/hyeon03-sketch/IML-Final-project.git"
REPO_BRANCH = "codex/ml-project-review"
REPO_DIR = Path("IML-Final-project")
DATA_FILE = Path("IML_Final_dataset.xlsx")

if not DATA_FILE.exists() and Path("../IML_Final_dataset.xlsx").exists():
    os.chdir("..")

if not DATA_FILE.exists():
    if not REPO_DIR.exists():
        subprocess.check_call(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL])
    os.chdir(REPO_DIR)

print("Working directory:", Path.cwd())
print("Dataset exists:", Path("IML_Final_dataset.xlsx").exists())

packages = {
    "pandas": "pandas",
    "numpy": "numpy",
    "openpyxl": "openpyxl",
    "sklearn": "scikit-learn",
    "xgboost": "xgboost",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
}
missing = [pip_name for import_name, pip_name in packages.items() if importlib.util.find_spec(import_name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("Installed:", missing)
else:
    print("All required packages are already installed.")

## 2. Imports and Korean Font Setup

In [ ]:
import warnings
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, GroupShuffleSplit, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
RND = 42
DATA_PATH = Path("IML_Final_dataset.xlsx")
SHEET_NAME = "최종데이터셋_모델용"
TARGET = "보증금(만원)"
OUT_DIR = Path("outputs_jeonse_only_no_scaler")
OUT_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 100)


def setup_korean_font():
    font_candidates = [
        ("NanumGothic", Path("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")),
        ("Apple SD Gothic Neo", Path("/System/Library/Fonts/AppleSDGothicNeo.ttc")),
        ("AppleGothic", Path("/Library/Fonts/AppleGothic.ttf")),
        ("Malgun Gothic", Path("C:/Windows/Fonts/malgun.ttf")),
    ]
    if not any(path.exists() for _, path in font_candidates):
        try:
            subprocess.run(["apt-get", "update", "-qq"], check=False)
            subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-nanum"], check=False)
            try:
                fm._load_fontmanager(try_read_cache=False)
            except Exception:
                pass
        except Exception as exc:
            print("Korean font installation skipped:", exc)
    for family, path in font_candidates:
        if path.exists():
            try:
                fm.fontManager.addfont(str(path))
            except Exception:
                pass
            sns.set_theme(style="whitegrid", font=family)
            mpl.rcParams["font.family"] = family
            mpl.rcParams["axes.unicode_minus"] = False
            print(f"Korean font configured: {family}")
            return family
    sns.set_theme(style="whitegrid")
    mpl.rcParams["axes.unicode_minus"] = False
    print("Korean font was not found. Korean chart labels may not render correctly.")
    return None

KOREAN_FONT = setup_korean_font()

## 3. Load Jeonse Rows Only

Monthly-rent rows are removed. The target is the original Jeonse deposit, `보증금(만원)`. No Jeonse-equivalent conversion is performed.

In [ ]:
df_all = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)
df = df_all[df_all["전월세구분"].eq("전세")].copy()
df = df[df[TARGET] > 0].copy()

print("Original rows:", len(df_all))
print("Jeonse-only rows:", len(df))
print("Removed monthly-rent rows:", len(df_all) - len(df))
print("Dong count:", df["읍면동"].nunique())
print("Contract years:", sorted(df["계약연도"].unique()))
print("Missing cells:", int(df.isna().sum().sum()))

display(df[[TARGET, "전용면적(㎡)", "층", "건물연령(계약기준)", "계약연도", "계약월"]].describe())
display(df.head())

## 4. EDA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df[TARGET], bins=50, ax=axes[0], color="#2563eb")
axes[0].set_title("Jeonse deposit distribution")
axes[0].set_xlabel("보증금(만원)")

sns.boxplot(x=df[TARGET], ax=axes[1], color="#059669")
axes[1].set_title("Jeonse deposit boxplot")
axes[1].set_xlabel("보증금(만원)")
plt.tight_layout()
plt.savefig(OUT_DIR / "target_distribution.png", dpi=150)
plt.show()

corr_cols = [
    TARGET, "전용면적(㎡)", "층", "건물연령(계약기준)", "계약연도", "계약월",
    "공원수", "최대공원면적(㎡)", "총공원면적(㎡)", "바다여부",
    "동_초등학교수", "동_중학교수", "동_고등학교수", "동_총학교수",
]
plt.figure(figsize=(11, 8))
sns.heatmap(df[corr_cols].corr(), cmap="RdBu_r", center=0, annot=True, fmt=".2f")
plt.title("Correlation Matrix: Jeonse Only")
plt.tight_layout()
plt.savefig(OUT_DIR / "correlation_matrix.png", dpi=150)
plt.show()

## 5. VIF Check and Feature Set

VIF is used to document multicollinearity. The default spatial feature set excludes `동_총학교수` and `총공원면적(㎡)`.

In [ ]:
def calculate_vif(data, columns):
    X = data[columns].astype(float).replace([np.inf, -np.inf], np.nan).dropna()
    rows = []
    for col in columns:
        y = X[col].to_numpy()
        others = [c for c in columns if c != col]
        X_other = X[others].to_numpy()
        X_design = np.column_stack([np.ones(len(X_other)), X_other])
        beta, *_ = np.linalg.lstsq(X_design, y, rcond=None)
        pred = X_design @ beta
        ss_res = np.sum((y - pred) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
        vif = np.inf if r2 >= 0.999999 else 1 / (1 - r2)
        rows.append({"feature": col, "vif": vif})
    return pd.DataFrame(rows).sort_values("vif", ascending=False)

candidate_vif_cols = [
    "전용면적(㎡)", "층", "건물연령(계약기준)", "계약연도", "계약월",
    "공원수", "최대공원면적(㎡)", "총공원면적(㎡)",
    "동_초등학교수", "동_중학교수", "동_고등학교수", "동_총학교수",
]
print("Candidate variables before pruning")
display(calculate_vif(df, candidate_vif_cols))

selected_vif_cols = [
    "전용면적(㎡)", "층", "건물연령(계약기준)", "계약연도", "계약월",
    "공원수", "최대공원면적(㎡)",
    "동_초등학교수", "동_중학교수", "동_고등학교수",
]
print("Default numeric variables after pruning")
display(calculate_vif(df, selected_vif_cols))

## 6. Feature Groups

Main comparison:

- `A_baseline`: housing and transaction-time variables only.
- `B_spatial_extended`: Group A plus selected spatial-derived variables.

Supplementary comparison adds `읍면동` as a location-control variable.

In [ ]:
HOUSE_TIME = ["전용면적(㎡)", "층", "건물연령(계약기준)", "계약연도", "계약월"]
DONG = ["읍면동"]
SEA = ["바다여부"]
SPATIAL = [
    "공원여부", "공원수", "최대공원면적(㎡)",
    "동_초등학교수", "동_중학교수", "동_고등학교수",
    "동_초중고모두있음여부",
]

BINARY_COLS = {"바다여부", "공원여부", "동_초중고모두있음여부"}
CATEGORICAL_COLS = {"읍면동"}

FEATURE_GROUPS = {
    "A_baseline": HOUSE_TIME,
    "B_spatial_extended": HOUSE_TIME + SEA + SPATIAL,
    "A_location_control_with_dong": HOUSE_TIME + DONG,
    "B_spatial_location_control_with_dong": HOUSE_TIME + DONG + SEA + SPATIAL,
}

for group_name, cols in FEATURE_GROUPS.items():
    print(group_name, len(cols), cols)

## 7. Modeling: No Scaler + GridSearchCV

No scaler is applied. Numeric and binary variables pass through as-is; only categorical variables are one-hot encoded.

In [ ]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def split_feature_types(cols):
    categorical = [c for c in cols if c in CATEGORICAL_COLS]
    binary = [c for c in cols if c in BINARY_COLS]
    numeric = [c for c in cols if c not in set(categorical + binary)]
    return numeric, categorical, binary


def make_preprocessor(cols):
    numeric, categorical, binary = split_feature_types(cols)
    transformers = []
    pass_cols = numeric + binary
    if pass_cols:
        transformers.append(("pass", "passthrough", pass_cols))
    if categorical:
        transformers.append(("cat", make_one_hot_encoder(), categorical))
    return ColumnTransformer(transformers=transformers, remainder="drop")


def make_models_and_grids():
    return {
        "Ridge": (
            Ridge(),
            {"model__alpha": [0.1, 1.0, 10.0, 100.0]},
        ),
        "RandomForest": (
            RandomForestRegressor(random_state=RND, n_jobs=-1),
            {
                "model__n_estimators": [300],
                "model__max_depth": [10, 20, None],
                "model__min_samples_leaf": [1, 3],
            },
        ),
        "XGBoost": (
            XGBRegressor(
                objective="reg:squarederror",
                tree_method="hist",
                random_state=RND,
                n_jobs=-1,
                importance_type="gain",
            ),
            {
                "model__n_estimators": [300, 600],
                "model__max_depth": [4, 6],
                "model__learning_rate": [0.05, 0.1],
                "model__subsample": [0.9],
                "model__colsample_bytree": [0.9],
            },
        ),
    }


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def mape_pct(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = y_true > 0
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)


def evaluate_group(data, group_name, cols):
    X = data[cols].copy()
    y = data[TARGET].copy()
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RND
    )

    rows = []
    fitted = {}
    for model_name, (model, grid) in make_models_and_grids().items():
        pipe = Pipeline([
            ("pre", make_preprocessor(cols)),
            ("model", model),
        ])
        search = GridSearchCV(
            pipe,
            param_grid=grid,
            scoring="neg_root_mean_squared_error",
            cv=KFold(n_splits=5, shuffle=True, random_state=RND),
            n_jobs=-1,
            verbose=0,
        )
        search.fit(X_train, y_train)
        pred = search.predict(X_test)
        rows.append({
            "experiment": group_name,
            "model": model_name,
            "n_rows": len(data),
            "n_features_raw": len(cols),
            "cv_rmse": float(-search.best_score_),
            "test_rmse": rmse(y_test, pred),
            "test_mae": float(mean_absolute_error(y_test, pred)),
            "test_mape_pct": mape_pct(y_test, pred),
            "test_r2": float(r2_score(y_test, pred)),
            "mae_pct_of_mean_target": float(mean_absolute_error(y_test, pred) / y_test.mean() * 100),
            "best_params": search.best_params_,
        })
        fitted[model_name] = {
            "estimator": search.best_estimator_,
            "X_test": X_test,
            "y_test": y_test,
            "pred": pred,
            "best_params": search.best_params_,
        }
    return pd.DataFrame(rows), fitted

## 8. Run Main Experiments

In [ ]:
all_results = []
all_fitted = {}

for group_name, cols in FEATURE_GROUPS.items():
    result, fitted = evaluate_group(df, group_name, cols)
    all_results.append(result)
    all_fitted[group_name] = fitted

results = pd.concat(all_results, ignore_index=True).sort_values(["experiment", "test_rmse"])
display(results)
results.to_csv(OUT_DIR / "model_results.csv", index=False, encoding="utf-8-sig")

## 9. Performance Comparison

MAPE is reported as a percentage error, which makes the absolute prediction error easier to interpret relative to the target size.


In [ ]:
best_by_group = (
    results.sort_values("test_rmse")
    .groupby("experiment", as_index=False)
    .first()[["experiment", "model", "cv_rmse", "test_rmse", "test_mae", "test_mape_pct", "test_r2", "mae_pct_of_mean_target", "best_params"]]
)
display(best_by_group)

rmse_pivot = results.pivot_table(index="model", columns="experiment", values="test_rmse")
display(rmse_pivot)

def compare_delta(left, right):
    merged = results[results["experiment"].isin([left, right])].pivot_table(
        index="model", columns="experiment", values=["test_rmse", "test_mae", "test_mape_pct", "test_r2"]
    )
    delta = pd.DataFrame(index=merged.index)
    delta["delta_rmse"] = merged[("test_rmse", right)] - merged[("test_rmse", left)]
    delta["delta_mae"] = merged[("test_mae", right)] - merged[("test_mae", left)]
    delta["delta_mape_pct"] = merged[("test_mape_pct", right)] - merged[("test_mape_pct", left)]
    delta["delta_r2"] = merged[("test_r2", right)] - merged[("test_r2", left)]
    return delta

print("Main methodology comparison: Group B - Group A")
display(compare_delta("A_baseline", "B_spatial_extended"))

print("Supplementary location-control comparison: Spatial with dong - Baseline with dong")
display(compare_delta("A_location_control_with_dong", "B_spatial_location_control_with_dong"))

## 10. Prediction vs Actual

In [ ]:
best_spatial_model = (
    results[results["experiment"].eq("B_spatial_extended")]
    .sort_values("test_rmse")
    .iloc[0]["model"]
)
r = all_fitted["B_spatial_extended"][best_spatial_model]

plt.figure(figsize=(6, 6))
plt.scatter(r["y_test"], r["pred"], s=10, alpha=0.45, color="#2563eb")
lim = max(r["y_test"].max(), np.max(r["pred"]))
plt.plot([0, lim], [0, lim], "r--", lw=1)
plt.xlabel("Actual Jeonse deposit (만원)")
plt.ylabel("Predicted Jeonse deposit (만원)")
plt.title(f"Prediction vs Actual: {best_spatial_model}")
plt.tight_layout()
plt.savefig(OUT_DIR / "prediction_vs_actual_best_spatial.png", dpi=150)
plt.show()

## 11. XGBoost Feature Importance

In [ ]:
group_name = "B_spatial_extended"
model_name = "XGBoost"
bundle = all_fitted[group_name][model_name]
pipe = bundle["estimator"]
pre = pipe.named_steps["pre"]
model = pipe.named_steps["model"]
feature_names = pre.get_feature_names_out()
importance = pd.DataFrame({
    "feature": feature_names,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)

display(importance.head(20))
importance.to_csv(OUT_DIR / "xgboost_feature_importance.csv", index=False, encoding="utf-8-sig")

top = importance.head(15).iloc[::-1]
plt.figure(figsize=(9, 6))
plt.barh(top["feature"], top["importance"], color="#059669")
plt.title("XGBoost Feature Importance: Jeonse Only, No Scaler")
plt.tight_layout()
plt.savefig(OUT_DIR / "xgboost_feature_importance_top15.png", dpi=150)
plt.show()

## 12. Optional Strict Validation

Set `RUN_STRICT_VALIDATION = True` to test time, complex, and dong holdout splits. This can take additional time because it uses GridSearchCV.

In [ ]:
RUN_STRICT_VALIDATION = False
STRICT_MODEL_NAME = "XGBoost"

if RUN_STRICT_VALIDATION:
    def metric_row(validation, experiment, model, y_true, pred, extra=None):
        row = {
            "validation": validation,
            "experiment": experiment,
            "model": model,
            "n_test": len(y_true),
            "test_rmse": rmse(y_true, pred),
            "test_mae": float(mean_absolute_error(y_true, pred)),
        "test_mape_pct": mape_pct(y_true, pred),
            "test_r2": float(r2_score(y_true, pred)),
        }
        if extra:
            row.update(extra)
        return row

    def fit_grid_on_holdout(data, group_name, cols, train_idx, test_idx):
        X = data[cols].copy()
        y = data[TARGET].copy()
        X_train, X_test = X.loc[train_idx], X.loc[test_idx]
        y_train, y_test = y.loc[train_idx], y.loc[test_idx]
        model, grid = make_models_and_grids()[STRICT_MODEL_NAME]
        pipe = Pipeline([("pre", make_preprocessor(cols)), ("model", model)])
        search = GridSearchCV(
            pipe, param_grid=grid, scoring="neg_root_mean_squared_error",
            cv=KFold(n_splits=5, shuffle=True, random_state=RND), n_jobs=-1,
        )
        search.fit(X_train, y_train)
        pred = search.predict(X_test)
        return y_train, y_test, pred, search.best_params_

    def run_holdout(validation, train_idx, test_idx):
        rows = []
        y_train = df.loc[train_idx, TARGET]
        y_test = df.loc[test_idx, TARGET]
        rows.append(metric_row(validation, "naive_baseline", "TrainMean", y_test, np.full(len(y_test), y_train.mean())))
        rows.append(metric_row(validation, "naive_baseline", "TrainMedian", y_test, np.full(len(y_test), np.median(y_train))))
        for group_name in ["A_baseline", "B_spatial_extended"]:
            _, y_test, pred, best_params = fit_grid_on_holdout(df, group_name, FEATURE_GROUPS[group_name], train_idx, test_idx)
            rows.append(metric_row(validation, group_name, STRICT_MODEL_NAME, y_test, pred, {"best_params": best_params}))
        return rows

    strict_rows = []
    train_idx, test_idx = train_test_split(df.index, test_size=0.2, random_state=RND)
    strict_rows.extend(run_holdout("random_split", train_idx, test_idx))

    years = sorted(df["계약연도"].dropna().unique())
    if len(years) >= 2:
        latest_year = years[-1]
        train_idx = df.index[df["계약연도"] < latest_year]
        test_idx = df.index[df["계약연도"] == latest_year]
        if len(train_idx) >= 100 and len(test_idx) >= 50:
            strict_rows.extend(run_holdout(f"time_holdout_test_{latest_year}", train_idx, test_idx))

    complex_groups = df["단지명"].fillna("missing_complex")
    train_pos, test_pos = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RND).split(df, groups=complex_groups))
    strict_rows.extend(run_holdout("complex_holdout", df.index[train_pos], df.index[test_pos]))

    dong_groups = df["읍면동"].fillna("missing_dong")
    train_pos, test_pos = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RND).split(df, groups=dong_groups))
    strict_rows.extend(run_holdout("dong_holdout", df.index[train_pos], df.index[test_pos]))

    strict_validation = pd.DataFrame(strict_rows).sort_values(["validation", "experiment", "test_rmse"])
    display(strict_validation)
    strict_validation.to_csv(OUT_DIR / "strict_validation_results.csv", index=False, encoding="utf-8-sig")
else:
    print("Strict validation skipped. Set RUN_STRICT_VALIDATION = True to run it.")